# Decompressing a Sentinel-2 scene: GeoTIFF vs Rumi

Same pixels, same machine, decompress only. The encoded bytes are loaded into RAM
before timing starts, so no page cache or GDAL block cache is in the measurement.

In [ ]:
!pip install -q "rumi-eo[write]" rasterio tifffile

In [ ]:
import os
os.environ["GDAL_CACHEMAX"] = "64"
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"

import hashlib, statistics as st, time
import numpy as np, rasterio, tifffile, geozl, rumi

THREADS = os.cpu_count()
print(f"rumi {rumi.__version__}  geozl {geozl.__version__}  "
      f"GDAL {rasterio.__gdal_version__}  {THREADS} cores")

## The scene

S2A_37MBV_20241029, true-colour, 318 MB on S3.

In [ ]:
URL = ("https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/"
       "37/M/BV/2024/10/S2A_37MBV_20241029_0_L2A/TCI.tif")
!wget -q -O TCI.tif {URL}

with rasterio.open("TCI.tif") as ds:
    print(ds.count, "x", ds.shape, ds.dtypes[0])
    print("blocks    ", ds.block_shapes[0])
    print("compress  ", ds.profile["compress"],
          "predictor", ds.tags(ns="IMAGE_STRUCTURE")["PREDICTOR"])
    print("interleave", ds.profile["interleave"])
    print("overviews ", ds.overviews(1))
    arr = ds.read()

RAW_MB = arr.nbytes / 1e6
print(f"\nuncompressed {RAW_MB:.1f} MB")

## Write it as Rumi

One frame per 1024x1024 tile holding all three bands. Two recipes: PFOR bit-packing,
which has no entropy coder and is built for decode speed, and a MED predictor with an
entropy coder, which trades that speed for a smaller file.

In [ ]:
TILE = 1024
RECIPES = {"planar>zigzag>pfor": "pfor.rumi", "med>zigzag>entropy": "medent.rumi"}

headers = {}
for recipe, path in RECIPES.items():
    frames = rumi.frames(arr, "b (row h) (col w) -> row col (b h w)", tile_size=TILE)
    graphs = {}
    t0 = time.perf_counter()
    for f in frames:
        g = graphs.get(f.data.shape)
        if g is None:
            g = graphs[f.data.shape] = geozl.graph(f.data, recipe)
        f.compressed = geozl.compress(f.data, graph=g)
    encode_s = time.perf_counter() - t0
    _, headers[recipe] = rumi.write(path, frames)
    print(f"{recipe:22s} {len(frames)} frames in {encode_s:5.1f}s")

### Size, counting the same pixels

The COG carries overview levels the Rumi files do not, so its size on disk is not the
number to compare against. Summing the first IFD's tile byte counts gives the full
resolution share.

In [ ]:
with tifffile.TiffFile("TCI.tif") as tif:
    COG_MB = sum(tif.pages[0].databytecounts) / 1e6
    overview_mb = sum(sum(p.databytecounts) for p in tif.pages[1:]) / 1e6

print(f"GeoTIFF on disk        {os.path.getsize('TCI.tif') / 1e6:6.1f} MB")
print(f"  full resolution      {COG_MB:6.1f} MB")
print(f"  overviews            {overview_mb:6.1f} MB\n")
for recipe, path in RECIPES.items():
    mb = os.path.getsize(path) / 1e6
    print(f"{recipe:22s} {mb:6.1f} MB   {100 * (mb - COG_MB) / COG_MB:+5.1f}% vs full resolution")

## Benchmark

Every file is read into RAM first. Each engine decodes five times; we keep the median.

In [ ]:
tif_bytes = open("TCI.tif", "rb").read()
rumi_bytes = {r: open(p, "rb").read() for r, p in RECIPES.items()}

def timeit(fn, n=5):
    times, digest = [], None
    for _ in range(n):
        t0 = time.perf_counter()
        out = np.asarray(fn())
        times.append(time.perf_counter() - t0)
        if digest is None:
            digest = hashlib.sha256(np.ascontiguousarray(out)).hexdigest()[:12]
        del out
    return st.median(times), digest

def gdal(driver, threads):
    mf = rasterio.MemoryFile(tif_bytes)
    ds = mf.open(driver=driver, NUM_THREADS=str(threads))
    return timeit(ds.read)

def rumi_read(recipe, threads):
    rumi.set_num_threads(threads)
    return timeit(lambda: rumi.read(rumi_bytes[recipe], headers[recipe]))

In [ ]:
engines = [("GDAL GTiff", lambda t: gdal("GTiff", t)),
           ("GDAL LIBERTIFF", lambda t: gdal("LIBERTIFF", t))]
engines += [(f"rumi {r.split('>')[0]}", (lambda r: lambda t: rumi_read(r, t))(r))
            for r in RECIPES]

results = {}
for threads in (1, THREADS):
    for label, run in engines:
        os.environ["GDAL_NUM_THREADS"] = str(threads)
        results[label, threads] = run(threads)

print(f"{'':16s} {'threads':>7} {'ms':>8} {'MB/s':>8}")
for (label, threads), (secs, _) in results.items():
    print(f"{label:16s} {threads:7d} {secs * 1000:8.1f} {RAW_MB / secs:8.0f}")

assert len({d for _, d in results.values()}) == 1, "engines disagree"
print("\nall engines produced identical pixels")

In [ ]:
for threads in (1, THREADS):
    base = results["GDAL LIBERTIFF", threads][0]
    for label, _ in engines[2:]:
        print(f"{threads:2d} threads: {label} is {base / results[label, threads][0]:4.2f}x "
              f"the speed of GDAL LIBERTIFF")

## Caveats

The GeoTIFF is pixel-interleaved and DEFLATE-compressed; the Rumi files store each tile
planar. Part of the gap is the codec and part is the layout. Colab gives you 2 vCPUs, so
the multi-threaded column is much less interesting than on a real machine.

Same notebook on an Apple M5 (4 performance + 6 efficiency cores):

| | size | 1 thread | 10 threads |
|---|---|---|---|
| GDAL GTiff | 237 MB | 1352 ms | 203 ms |
| GDAL LIBERTIFF | 237 MB | 722 ms | 113 ms |
| rumi, PFOR | 250 MB | 146 ms | 40 ms |
| rumi, MED + entropy | 210 MB | 457 ms | 106 ms |
